# 06 — Longitudinal Modeling (R Kernel)
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: Synthetic data only. Requires R kernel (IRkernel).

## Goals
1. Fit unconditional linear LGCM for RSA trajectories (lavaan)
2. Add time-invariant predictors (GA, group)
3. Multigroup CFA to test group differences in growth
4. Mixed-effects model for HDA-SA with lme4
5. Publication-quality trajectory plot with ggplot2

> **Note**: If running without R kernel, this notebook demonstrates R code structurally — output will not be rendered.
> Install IRkernel: `install.packages('IRkernel'); IRkernel::installspec()`

In [ ]:
# R kernel setup
options(warn = -1)
suppressPackageStartupMessages({
  library(lavaan)
  library(lme4)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})
cat('Packages loaded.\n')
cat(sprintf('lavaan %s | lme4 %s\n', packageVersion('lavaan'), packageVersion('lme4')))

## 1. Generate Synthetic Longitudinal RSA Data

In [ ]:
set.seed(42)
N <- 180
timepoints <- c(1, 3, 6, 12, 36)

groups <- sample(c('ASIB', 'PT', 'TD'), N, replace = TRUE, prob = c(0.20, 0.50, 0.30))
ga_weeks <- rnorm(N, 29, 2)
sex <- rbinom(N, 1, 0.5)

# Growth parameters
intercepts <- rnorm(N, mean = ifelse(groups == 'ASIB', 4.5, ifelse(groups == 'PT', 4.2, 4.0)),
                    sd = 0.7)
slopes <- rnorm(N, mean = ifelse(groups == 'ASIB', -0.06, ifelse(groups == 'PT', -0.04, -0.03)),
                sd = 0.015)

df_wide <- data.frame(
  id      = paste0('NANO-', sprintf('%04d', 1:N)),
  group   = groups,
  ga_weeks = round(ga_weeks, 1),
  sex     = sex
)

for (i in seq_along(timepoints)) {
  tp <- timepoints[i]
  col_name <- paste0('rsa_t', i)
  df_wide[[col_name]] <- intercepts + slopes * tp + rnorm(N, 0, 0.35)
  # Introduce ~10% missingness increasing over time
  miss_idx <- sample(1:N, size = round(0.05 * i * N))
  df_wide[miss_idx, col_name] <- NA
}

cat(sprintf('Wide dataset: %d participants x %d columns\n', nrow(df_wide), ncol(df_wide)))
head(df_wide)

## 2. Unconditional Linear LGCM (lavaan)

In [ ]:
# Time coding: 0, 1, 2, 3, 4 (equal intervals for demo)
lgcm_model <- '
  # Latent intercept
  i =~ 1*rsa_t1 + 1*rsa_t2 + 1*rsa_t3 + 1*rsa_t4 + 1*rsa_t5
  # Latent slope (time coded 0–4)
  s =~ 0*rsa_t1 + 1*rsa_t2 + 2*rsa_t3 + 3*rsa_t4 + 4*rsa_t5
  # Means
  i ~ 1
  s ~ 1
  # (Co)variances
  i ~~ i
  s ~~ s
  i ~~ s
'

fit_lgcm <- growth(lgcm_model, data = df_wide, missing = 'fiml', estimator = 'MLR')
summary(fit_lgcm, fit.measures = TRUE, standardized = TRUE)

In [ ]:
# Extract key fit indices
fit_idx <- fitMeasures(fit_lgcm, c('rmsea', 'rmsea.ci.lower', 'rmsea.ci.upper', 'cfi', 'srmr', 'chisq', 'df', 'pvalue'))
cat(sprintf('\nModel Fit Indices:\n'))
cat(sprintf('  χ²(%d) = %.2f, p = %.3f\n', fit_idx['df'], fit_idx['chisq'], fit_idx['pvalue']))
cat(sprintf('  RMSEA = %.3f [90%% CI: %.3f-%.3f]\n', fit_idx['rmsea'], fit_idx['rmsea.ci.lower'], fit_idx['rmsea.ci.upper']))
cat(sprintf('  CFI = %.3f\n', fit_idx['cfi']))
cat(sprintf('  SRMR = %.3f\n', fit_idx['srmr']))

## 3. Mixed-Effects Model (lme4) — Long Format

In [ ]:
# Convert to long format
df_long <- df_wide |>
  pivot_longer(cols = starts_with('rsa_t'), names_to = 'timepoint', values_to = 'rsa') |>
  mutate(
    time_num = as.integer(sub('rsa_t', '', timepoint)) - 1,
    group = factor(group, levels = c('TD', 'PT', 'ASIB')),
    sex = factor(sex, labels = c('Male', 'Female'))
  ) |>
  filter(!is.na(rsa))

cat(sprintf('Long format: %d rows\n', nrow(df_long)))

# Random intercept + slope model
lmer_fit <- lmer(rsa ~ time_num * group + ga_weeks + sex + (1 + time_num | id),
                  data = df_long, REML = TRUE)
summary(lmer_fit)

## 4. Trajectory Plot (ggplot2)

In [ ]:
# Mean ± SE per group × timepoint
df_summary <- df_long |>
  group_by(group, time_num) |>
  summarise(
    mean_rsa = mean(rsa, na.rm = TRUE),
    se_rsa   = sd(rsa, na.rm = TRUE) / sqrt(sum(!is.na(rsa))),
    n = sum(!is.na(rsa)),
    .groups = 'drop'
  ) |>
  mutate(
    ci_lo = mean_rsa - 1.96 * se_rsa,
    ci_hi = mean_rsa + 1.96 * se_rsa,
    month_label = c('1', '3', '6', '12', '36')[time_num + 1]
  )

group_colors <- c('ASIB' = '#E41A1C', 'PT' = '#377EB8', 'TD' = '#4DAF4A')

p <- ggplot(df_summary, aes(x = time_num, y = mean_rsa, color = group, fill = group)) +
  geom_ribbon(aes(ymin = ci_lo, ymax = ci_hi), alpha = 0.15, color = NA) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_color_manual(values = group_colors) +
  scale_fill_manual(values = group_colors) +
  scale_x_continuous(breaks = 0:4, labels = c('1m', '3m', '6m', '12m', '36m')) +
  labs(
    x = 'Visit (corrected age)', y = 'RSA (log power)',
    title = 'RSA Developmental Trajectories by Group',
    subtitle = 'NANO Study — Synthetic Data Demo | Mean ± 95% CI',
    color = 'Group', fill = 'Group',
    caption = 'ASIB = autism siblings; PT = preterm; TD = typically developing'
  ) +
  theme_classic(base_size = 13) +
  theme(
    legend.position = 'top',
    plot.title = element_text(face = 'bold')
  )
print(p)

## Summary
- Linear LGCM fit with FIML estimation to handle missing data
- Random intercept + slope lmer model tests group × time interaction
- RSA shows declining trajectories, with ASIB group having highest initial values (by design in synthetic data)

**Next**: See `07_redcap_api_demo.ipynb` for REDCap API pull/push demonstration.